# 14 — Telco: A first local prediction service
**Preview of next semester:** This lesson is available in advance of the dedicated
**MLOps & Model Deployment** course. Production deployment, infrastructure, automation
and operational monitoring will be covered there.

**Session 8 · Objective:** Serve a trained Telco pipeline on your own computer and
obtain a churn probability through an HTTP request. Follow the lesson's sequence:
preserve the pipeline, reload it in another process, validate a request, score and respond,
then ask what we would watch as the population changes.

The complete terminal walkthrough is in [`serving/README.md`](../serving/README.md).
This notebook also runs independently from `notebooks/`. It does not need notebook
state or instructor files. It prepares and reloads the model automatically; the HTTP
cells need a Uvicorn server running in a separate terminal, as explained below.

## Install and keep the fitted preparation
Create and activate `.venv` using the root README. In a terminal **at the repository root**:

```bash
python -m pip install -r serving/requirements.txt
python -m pip check
```

Choose that same environment as the notebook kernel. FastAPI routes HTTP requests,
Pydantic describes and validates their bodies, Uvicorn runs the local server, and
joblib saves and restores the fitted object. Requests use Python's standard library.

Keep the complete Session 6 finalist: two row-wise logs, training medians and scales,
one-hot categories and Logistic Regression. Session 6 selected raw probabilities on
validation; we refit the fixed recipe on the same 5,634 development customers and
leave the classroom holdout out of training. This does not add new performance evidence.

In [1]:
# Locate the repository and its saved customer example across operating systems.
from pathlib import Path
# Read customer JSON and serialize request and response bodies.
import json
# Run the same training and client commands a student uses in a separate terminal.
import subprocess
# Use this notebook kernel's Python interpreter for all child processes.
import sys
# Print installed distribution versions for this particular notebook environment.
from importlib.metadata import version
# Inspect validation-error responses instead of treating them as missing server replies.
from urllib.error import HTTPError
# Send real HTTP requests using only the Python standard library.
from urllib.request import Request, urlopen

# Move one level above notebooks/ so module commands and data paths start at the repository root.
root = Path.cwd().parent
# Stop early if the working directory does not point to the expected repository layout.
assert (root / 'serving' / 'train.py').exists(), 'Run this notebook from notebooks/.'
# Check that course and serving dependencies are visible to the selected kernel.
for package in ['scikit-learn', 'pandas', 'joblib', 'fastapi', 'pydantic', 'uvicorn']:
    # Expose each installed version alongside the saved serving example.
    print(package, version(package))

scikit-learn 1.9.1
pandas 3.0.6
joblib 1.6.0
fastapi 0.163.1
pydantic 2.14.3
uvicorn 0.54.0


## Prepare the artifact
In a terminal at the root the command is `python -m serving.train`. The cell below
runs that exact module with the kernel's interpreter. Inspect
[`pipeline.py`](../serving/pipeline.py) and [`train.py`](../serving/train.py): the transform
`add_logs` lives in an importable module so a new Python process can recover it.
An object saved from a notebook-only function can refer to an unavailable `__main__`.

In [2]:
# Train once outside the notebook process; generated artifacts are ignored by Git.
# sys.executable selects this kernel's Python; -m runs the importable training module.
# cwd sets the child process directory; check=True raises if training fails.
# capture_output keeps its messages in result.stdout, and text=True decodes them as text.
result = subprocess.run([sys.executable, '-m', 'serving.train'], cwd=root,
                         check=True, capture_output=True, text=True)
# Display training counts and artifact metadata captured from the separate training process.
print(result.stdout)

Saved telco.joblib: 5,634 development customers
{
  "model_version": "telco-s6-v1",
  "training_customers": 5634,
  "dataset_sha256": "88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a",
  "python": "3.12.9",
  "packages": {
    "scikit-learn": "1.9.1",
    "numpy": "2.5.3",
    "pandas": "3.0.6",
    "scipy": "1.18.1",
    "joblib": "1.6.0"
  }
}



The artifact is `serving/artifacts/telco.joblib`; the adjacent JSON records dataset
hash, training count and versions. Use the same environment for loading, or regenerate
the artifact after changing model libraries. Load only trusted joblib files, since loading
can execute Python. See [model persistence](https://scikit-learn.org/stable/model_persistence.html).

## Same customer, new process
The client's JSON uses all 19 raw fields. Charges are in euros and tenure in months.
The Pipeline owns logs, imputation and encoding; the client sends neither derived
columns, the target nor the identifier. This is the exact class example.

In [3]:
# Load all nineteen raw fields as Python values, retaining JSON numbers and category strings.
customer = json.loads((root / 'serving' / 'customer.json').read_text(encoding='utf-8'))
# Inspect the complete body that will be sent, including field names and units described above.
print(json.dumps(customer, indent=2))

{
  "gender": "Male",
  "SeniorCitizen": 1,
  "Partner": "No",
  "Dependents": "No",
  "tenure": 23,
  "PhoneService": "No",
  "MultipleLines": "No phone service",
  "InternetService": "DSL",
  "OnlineSecurity": "No",
  "OnlineBackup": "No",
  "DeviceProtection": "No",
  "TechSupport": "No",
  "StreamingTV": "No",
  "StreamingMovies": "Yes",
  "Contract": "Month-to-month",
  "PaperlessBilling": "Yes",
  "PaymentMethod": "Bank transfer (automatic)",
  "MonthlyCharges": 34.65,
  "TotalCharges": 768.45
}


In [4]:
# Loading in a separate interpreter checks that custom transforms really are importable.
reload_code = """
# Decode the same customer file in the new interpreter, without relying on notebook variables.
import json
# Locate the files relative to the repository root used by the child process.
from pathlib import Path
# Restore the fitted Python object, including its reference to the importable log transform.
import joblib
# Construct the one-row labelled input expected by the saved Pipeline.
import pandas as pd
# Read the original raw fields rather than supplying manually preprocessed features.
customer = json.loads(Path('serving/customer.json').read_text())
# Load the already fitted preparation and classifier; do not train again.
pipeline = joblib.load('serving/artifacts/telco.joblib')
# Wrap the dictionary as one customer and emit only the positive-class probability.
print(float(pipeline.predict_proba(pd.DataFrame([customer]))[0, 1]))
"""
# Execute the reload program in this environment at the root; check=True raises on failure.
result = subprocess.run([sys.executable, '-c', reload_code], cwd=root,
                         check=True, capture_output=True, text=True)
# Parse the child process's numerical result as the reference for the later HTTP response.
expected_probability = float(result.stdout.strip())
# Show that a fresh process can recover and use the saved Pipeline.
print(f'Fresh-process probability: {expected_probability:.8f}')

Fresh-process probability: 0.35981820


Expected risk is approximately **0.3598182**, or 36%. The final decimals may vary with
library versions. Reusing learned preprocessing is what preserves the prediction.
Refitting a median on today's requests could change the answer for an unchanged customer.

## Start a local service
In a separate terminal, activate the same `.venv` and run **from the repository root**:

```bash
python -m uvicorn serving.app:app --host 127.0.0.1 --port 8000
```

Wait for `Application startup complete` and leave the terminal running while executing
the remaining cells. The server binds to your own computer. Stop it afterwards with
`Ctrl+C`. If port 8000 is busy, stop the other service or change both server and client ports.

Read [`app.py`](../serving/app.py): FastAPI's
[`lifespan`](https://fastapi.tiangolo.com/advanced/events/) loads the saved Pipeline **once per
server process**, before requests arrive. `POST /predict` builds a one-row DataFrame,
applies `predict_proba` and returns `churn_probability` plus `model_version`. It never
fits on incoming customers. Missing artifacts fail at startup with the preparation command.

Open <http://127.0.0.1:8000/docs> to see all schema fields. There is no `GET /` route.

## What is the API validating?
[`TelcoInput`](../serving/app.py) is a Pydantic model for all 19 raw fields. `strict=True`
rejects numeric strings such as `"34.65"`; `extra="forbid"` catches unexpected fields.
`Contract` must be `Month-to-month`, `One year` or `Two year`. All fields are required;
`TotalCharges` can explicitly be `null`, which becomes numeric `NaN` before the fitted
median handles it. Non-finite charges are rejected.

Tenure is an integer in [0, 72], monthly charges a number in [0, 200] and total charges
in [0, 20000]. These classroom guardrails and categories define the supported input
contract; they must be reviewed for a broader population. They do not check all
cross-field relationships or tell us whether a bill was recorded correctly. Euros
and cents can share a valid JSON number while having different meaning.

For a Pydantic parameter, FastAPI [validates the body before the route runs](https://fastapi.tiangolo.com/tutorial/body/).
Valid structure lets scoring proceed; a missing required field gives HTTP 422 and no
model prediction. This is separate from judging whether the model is useful.

In [5]:
# Keep request creation and response decoding identical for valid and invalid examples.
def post_customer(payload):
    # JSON encodes the client's raw values; Content-Type tells the server their format.
    request = Request('http://127.0.0.1:8000/predict',
                      data=json.dumps(payload).encode('utf-8'),
                      headers={'Content-Type': 'application/json'}, method='POST')
    # Attempt the call while keeping error responses available for inspection.
    try:
        # Send the request and bound waiting time if the local server is not responding.
        response = urlopen(request, timeout=10)
    # HTTP 422 is returned as an exception object that still contains the server's response.
    except HTTPError as error:
        # Validation errors are real HTTP responses whose JSON body we can inspect.
        response = error
    # Close the response stream automatically after reading its status and body.
    with response:
        # Return the HTTP status separately from the decoded JSON so both can be checked.
        return response.status, json.load(response)

# Send the unchanged complete customer to the separately running Uvicorn service.
status, prediction = post_customer(customer)
# Show whether the server accepted the request before reading the predicted value.
print('HTTP', status)
# Display the response probability and model label in a readable JSON body.
print(json.dumps(prediction, indent=2))
# Require a successful prediction response, not merely a body that can be decoded.
assert status == 200
# Check HTTP scoring against direct fresh-process scoring of the same saved artifact.
assert abs(prediction['churn_probability'] - expected_probability) < 1e-12

HTTP 200
{
  "churn_probability": 0.3598181970496265,
  "model_version": "telco-s6-v1"
}


The returned JSON has `churn_probability` approximately `0.3598182` and `model_version`
`telco-s6-v1`. It matches direct scoring of the same saved object in a fresh process.
The label identifies the teaching recipe; the metadata sidecar identifies your local build.

**A probability is not an offer decision.** Under the original Session 6 assumptions,
the caller considers an offer at risk at least **0.20**, so this customer crosses that
cutoff. At 0.40 the same risk would not trigger an offer. A 20% risk cutoff is not a
20% contact cap. The risk does not estimate whether the offer will persuade this
customer to stay; that needs intervention evidence. Cost or capacity changes can
require a different policy without changing the predicted risk.

In [6]:
# Remove a required field while keeping every other value unchanged.
missing_contract = customer.copy()
# Remove just the contract so missing-field validation is isolated from other changes.
del missing_contract['Contract']
# Send the incomplete body through the same HTTP route as the valid request.
status, error = post_customer(missing_contract)
# Show the validation status rather than interpreting the body as a prediction.
print('HTTP', status)
# Inspect the error location, type and message returned by FastAPI.
print(json.dumps(error, indent=2))
# Require the input-validation response for this incomplete customer.
assert status == 422
# Check that the failure specifically names the missing Contract field in the JSON body.
assert any(item['loc'] == ['body', 'Contract'] and item['type'] == 'missing'
           for item in error['detail'])

HTTP 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "Contract"
      ],
      "msg": "Field required",
      "input": {
        "gender": "Male",
        "SeniorCitizen": 1,
        "Partner": "No",
        "Dependents": "No",
        "tenure": 23,
        "PhoneService": "No",
        "MultipleLines": "No phone service",
        "InternetService": "DSL",
        "OnlineSecurity": "No",
        "OnlineBackup": "No",
        "DeviceProtection": "No",
        "TechSupport": "No",
        "StreamingTV": "No",
        "StreamingMovies": "Yes",
        "PaperlessBilling": "Yes",
        "PaymentMethod": "Bank transfer (automatic)",
        "MonthlyCharges": 34.65,
        "TotalCharges": 768.45
      }
    }
  ]
}


Read `loc: ["body", "Contract"]`: validation identifies the missing field. The response
has error details instead of a churn probability. Add the field and send again; do not
repair missing categories by silently inventing them inside the route.

In [7]:
# These are the other invalid requests in the class demonstration.
for field, value in [('MonthlyCharges', '34.65'), ('Contract', 'Flex 2027')]:
    # Copy the valid customer so each failure changes only one field.
    invalid = customer.copy()
    # Inject either a numeric string or a category outside the agreed input contract.
    invalid[field] = value
    # Ask the live service to validate this altered body before scoring.
    status, error = post_customer(invalid)
    # Read the field-specific explanation for why this request was rejected.
    print(field, 'HTTP', status, error['detail'][0]['msg'])
    # Check that both malformed inputs fail validation instead of producing a risk.
    assert status == 422

MonthlyCharges HTTP 422 Input should be a valid number
Contract HTTP 422 Input should be 'Month-to-month', 'One year' or 'Two year'


## Run from another client
With the server still running, a second activated terminal at the root can run:

```bash
python -m serving.client
python -m serving.check_api
```

The check compares HTTP with the saved model and also checks a missing contract,
wrong type, unknown category and fitted imputation for `TotalCharges: null`.
No additional testing dependency is needed. Alternatively, macOS/Linux:

```bash
curl -i -X POST http://127.0.0.1:8000/predict   -H 'Content-Type: application/json'   --data-binary @serving/customer.json
```

Windows PowerShell (use `curl.exe` to avoid the older alias):

```powershell
curl.exe -i -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" --data-binary "@serving/customer.json"
```

The JSON file displayed above is the complete request body; omitting the other raw
fields would fail validation. All terminal commands assume the repository root.

In [8]:
# Use real HTTP through a separate process, just like the terminal instructions.
for module in ['serving.client', 'serving.check_api']:
    # Run each module from the root in this environment and raise if its HTTP checks fail.
    result = subprocess.run([sys.executable, '-m', module], cwd=root,
                             check=True, capture_output=True, text=True)
    # Expose the client response and validation-check outcomes instead of hiding successful execution.
    print(result.stdout)

HTTP 200
{
  "churn_probability": 0.3598181970496265,
  "model_version": "telco-s6-v1"
}



Valid customer: HTTP 200, probability 0.35981820; matches saved pipeline
Missing Contract: HTTP 422, required field identified
Invalid MonthlyCharges: HTTP 422
Invalid Contract: HTTP 422
Null TotalCharges: HTTP 200, matches fitted imputation



## A bridge to monitoring and drift
The Session 8 lesson separates three clocks. **Today**, we can inspect schema errors,
missingness, failures, latency and input distributions. **After outcomes mature**, we
can check ranking, calibration and detection by contract group, accounting for offers.
**With a randomized comparison**, we can estimate retention and incremental net value.

**Data drift** changes inputs, for example the share of monthly contracts. **Concept
drift** changes the relationship between comparable profiles and churn. Input changes
alone cannot establish concept drift. A mix change can alter aggregate metrics without
changing performance within either contract group.

The lesson introduces KS as a comparison of two cumulative numeric distributions and
PSI as changes of proportions in fixed bins. KS significance depends on sample size
and assumptions; PSI depends on binning and empty-cell treatment. Neither directly
measures lost value or implies automatic retraining. Investigate the feed, mix, scores
and offer economics first.

A future prediction service and CRM could link their events through a request ID,
model and policy versions, timestamps and permitted monitoring fields. Recording an
action as well as a score helps interpret subsequent outcomes. Operational logging,
drift jobs, alerts, production hosting, infrastructure, automation and retraining are
reserved for next semester's dedicated course.

**Exercise:** Retention reports that the campaign stopped paying. Which evidence would
distinguish a broken billing feed, changed customer risk and a more expensive offer?

The local lesson uses the Session 8 HTML's full JSON example and serving sequence;
only this repository is required to run it. For the full cross-platform instructions
and troubleshooting, return to [`serving/README.md`](../serving/README.md).